In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from collections import Counter

In [3]:
# Sliding window parameters
window_size = 128
step_size = 64
selected_labels = ['Walking', 'Jogging', 'Sitting', 'Standing']
selected_labels


['Walking', 'Jogging', 'Sitting', 'Standing']

In [4]:
# read CSV file
df = pd.read_csv("time_series_data_human_activities.csv")

# Maintain the target action
df = df[df['activity'].isin(selected_labels)]

# Sorting
df = df.sort_values(['user', 'timestamp'])
df.head()

,user,activity,timestamp,x-axis,y-axis,z-axis
0,1,Walking,4991922345000,0.69,10.80,-2.03
1,1,Walking,4991972333000,6.85,7.44,-0.50
2,1,Walking,4992022351000,0.93,5.63,-0.50
3,1,Walking,4992072339000,-2.11,5.01,-0.69
4,1,Walking,4992122358000,-4.59,4.29,-1.95


In [5]:
# label coding
le = LabelEncoder()
df['label'] = le.fit_transform(df['activity'])

# presenting the label
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label：", label_map)


Label： {'Jogging': 0, 'Sitting': 1, 'Standing': 2, 'Walking': 3}


In [6]:
segments = []
labels = []

for user in df['user'].unique():
    user_df = df[df['user'] == user]
    for start in range(0, len(user_df) - window_size, step_size):
        window = user_df.iloc[start:start + window_size]
        if len(window) == window_size:
            segment = window[['x-axis', 'y-axis', 'z-axis']].values
            label = Counter(window['label']).most_common(1)[0][0]
            segments.append(segment)
            labels.append(label)

# Convert to NumPy format
X = np.array(segments)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (13240, 128, 3)
y shape: (13240,)


In [7]:
np.save("X.npy", X)
np.save("y.npy", y)
